In [68]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [31]:
data = pd.read_csv(
    'Fraud.csv',
    dtype={
        'type': 'category'
    },
    usecols=['step','type','amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest','isFraud']
)

In [32]:
for col in ['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']:
    data[col] = data[col].astype('float32')

In [33]:
data.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 8 columns):
 #   Column          Dtype   
---  ------          -----   
 0   step            int64   
 1   type            category
 2   amount          float32 
 3   oldbalanceOrg   float32 
 4   newbalanceOrig  float32 
 5   oldbalanceDest  float32 
 6   newbalanceDest  float32 
 7   isFraud         int64   
dtypes: category(1), float32(5), int64(2)
memory usage: 224.5 MB


In [34]:
data.columns

Index(['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig',
       'oldbalanceDest', 'newbalanceDest', 'isFraud'],
      dtype='object')

In [35]:
data.head(5)

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.639648,170136.0,160296.359375,0.0,0.0,0
1,1,PAYMENT,1864.280029,21249.0,19384.720703,0.0,0.0,0
2,1,TRANSFER,181.000000,181.0,0.000000,0.0,0.0,1
3,1,CASH_OUT,181.000000,181.0,0.000000,21182.0,0.0,1
4,1,PAYMENT,11668.139648,41554.0,29885.859375,0.0,0.0,0


In [36]:
data['balance_error'] = (data['oldbalanceOrg'] - data['newbalanceOrig'] + data['amount'])
+ (data['newbalanceDest'] - data['oldbalanceDest'] - data['amount'])


0         -9.839640e+03
1         -1.864280e+03
2         -1.810000e+02
3         -2.136300e+04
4         -1.166814e+04
               ...     
6362615    0.000000e+00
6362616   -6.311410e+06
6362617   -5.000000e-01
6362618   -8.500025e+05
6362619    0.000000e+00
Length: 6362620, dtype: float32

In [37]:
data['balance_error'].sum()

np.float32(1.00931096e+12)

In [38]:
data['is_balance_mismatch'] = ((data['oldbalanceOrg'] - data['newbalanceOrig'] != data['amount'])).astype(int)

In [48]:
print("I identified inconsistencies in transaction balance flows where transferred amounts were not reflected in destination accounts,\n which strongly correlated with fraudulent behavior")

I identified inconsistencies in transaction balance flows where transferred amounts were not reflected in destination accounts,
 which strongly correlated with fraudulent behavior


In [39]:
data[data['isFraud'] == 1]

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_error,is_balance_mismatch
2,1,TRANSFER,181.000,181.000,0.0,0.000000e+00,0.000,1,362.00,0
3,1,CASH_OUT,181.000,181.000,0.0,2.118200e+04,0.000,1,362.00,0
251,1,TRANSFER,2806.000,2806.000,0.0,0.000000e+00,0.000,1,5612.00,0
252,1,CASH_OUT,2806.000,2806.000,0.0,2.620200e+04,0.000,1,5612.00,0
680,1,TRANSFER,20128.000,20128.000,0.0,0.000000e+00,0.000,1,40256.00,0
...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.125,339682.125,0.0,0.000000e+00,339682.125,1,679364.25,0
6362616,743,TRANSFER,6311409.500,6311409.500,0.0,0.000000e+00,0.000,1,12622819.00,0
6362617,743,CASH_OUT,6311409.500,6311409.500,0.0,6.848884e+04,6379898.000,1,12622819.00,0
6362618,743,TRANSFER,850002.500,850002.500,0.0,0.000000e+00,0.000,1,1700005.00,0


In [40]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 10 columns):
 #   Column               Dtype   
---  ------               -----   
 0   step                 int64   
 1   type                 category
 2   amount               float32 
 3   oldbalanceOrg        float32 
 4   newbalanceOrig       float32 
 5   oldbalanceDest       float32 
 6   newbalanceDest       float32 
 7   isFraud              int64   
 8   balance_error        float32 
 9   is_balance_mismatch  int64   
dtypes: category(1), float32(6), int64(3)
memory usage: 297.3 MB


In [41]:
data.describe()

,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_error,is_balance_mismatch
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,2.433972e+02,1.798619e+05,8.338831e+05,8.551136e+05,1.100702e+06,1.224996e+06,1.290820e-03,1.586313e+05,9.298787e-01
std,1.423320e+02,6.038582e+05,2.888243e+06,2.924048e+06,3.399180e+06,3.674129e+06,3.590480e-02,6.358249e+05,2.553513e-01
min,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-3.812500e+00,0.000000e+00
25%,1.560000e+02,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.001237e+03,1.000000e+00
50%,2.390000e+02,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,2.517371e+04,1.000000e+00
75%,3.350000e+02,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,1.643050e+05,1.000000e+00
max,7.430000e+02,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,9.244552e+07,1.000000e+00


In [42]:
data['isFraud'].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [45]:
# Fraud vs non fraud 
data.groupby('isFraud')['amount'].mean()

isFraud
0    1.781970e+05
1    1.467967e+06
Name: amount, dtype: float32

In [56]:
print(pd.crosstab(data['type'], data['isFraud']))

"""
I found that fraudulent transactions were predominantly of the 'TRANSFER' and 'CASH_OUT' types, 
indicating that these transaction types are more susceptible to fraud in this dataset.
"""

isFraud         0     1
type                   
CASH_IN   1399284     0
CASH_OUT  2233384  4116
DEBIT       41432     0
PAYMENT   2151495     0
TRANSFER   528812  4097


"\nI found that fraudulent transactions were predominantly of the 'TRANSFER' and 'CASH_OUT' types, \nindicating that these transaction types are more susceptible to fraud in this dataset.\n"

In [59]:
# Pattern Check 
data[data['isFraud']==1][['oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']]

,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
2,181.000,0.0,0.000000e+00,0.000
3,181.000,0.0,2.118200e+04,0.000
251,2806.000,0.0,0.000000e+00,0.000
252,2806.000,0.0,2.620200e+04,0.000
680,20128.000,0.0,0.000000e+00,0.000
...,...,...,...,...
6362615,339682.125,0.0,0.000000e+00,339682.125
6362616,6311409.500,0.0,0.000000e+00,0.000
6362617,6311409.500,0.0,6.848884e+04,6379898.000
6362618,850002.500,0.0,0.000000e+00,0.000


In [66]:
# balance diff 
data['org_diff'] = data['oldbalanceOrg'] - data['newbalanceOrig']
data['dest_diff'] = data['newbalanceDest'] - data['oldbalanceDest']
# error features
data['balance_error'] = data['org_diff'] - data['dest_diff'] + data['amount']
# suspicious flag 
data['is_larg_txn'] = (data['amount'] > 200000).astype(int)

In [67]:
data = pd.get_dummies(data, columns=['type'], drop_first=True)